# Demo — Orquestador multi-agente de due diligence de startups de IA

**Pre-entrega 6 · AI Engineering (Coderhouse).** Este notebook muestra el **flujo de delegación** con un LLM real
(Gemini 3.5 Flash Lite) sobre tres startups ficticias, cada una con su data room:

| Caso | Startup | Qué demuestra |
|---|---|---|
| 1 | **VozActiva** (agentes de voz) | Delegación investigador → analista → crítico; el contraste determinístico y el crítico sacan a la luz lo que el pitch no muestra. |
| 2 | **Contralia** (revisión de contratos) | Una startup sólida: el mismo flujo llega a otra recomendación. |
| 3 | **Aula Viva** (tutor con IA) | Data room incompleto: la rúbrica manda a buscar de nuevo y el sistema **se abstiene** en lugar de inventar. |

Cada paso imprime qué decidió el supervisor (y por qué) y qué aportó cada especialista.
Para re-ejecutarlo hace falta un `.env` con `GEMINI_API_KEY`
(`jupyter nbconvert --to notebook --execute --inplace demo.ipynb`; la ejecución guardada usó 59 llamadas a Flash Lite).

In [1]:
import logging

from dotenv import load_dotenv
from IPython.display import Markdown, display

from llm import settings_from_env
from logging_setup import configurar_logging
from orquestador import config_analisis, crear_orquestador
from tracing import analizar

load_dotenv()
configurar_logging(logging.WARNING)  # solo avisos: el recorrido se imprime abajo
grafo, corpus, settings = crear_orquestador()
llm = settings_from_env()
print(f"Modelo: {llm.provider} / {llm.model} (thinking {llm.thinking_level}, {llm.requests_per_minute:g} requests/min)")
print(f"Startups con data room: {corpus.startups}")
print(f"Frenos: {settings.max_pasos} pasos del supervisor; intentos por especialista "
      f"{settings.max_intentos_investigador}/{settings.max_intentos_analista}/{settings.max_intentos_critico}")

Modelo: gemini / gemini-3.5-flash-lite (thinking low, 12 requests/min)
Startups con data room: {'aulaviva': 'Aula Viva', 'contralia': 'Contralia', 'vozactiva': 'VozActiva'}
Frenos: 12 pasos del supervisor; intentos por especialista 3/3/2


## El grafo

Topología jerárquica: el supervisor rutea con una arista condicional (`Literal`), los especialistas siempre
vuelven a él y el validador es la compuerta antes del informe. El diagrama Mermaid está en el README.

In [2]:
print(grafo.get_graph().draw_ascii())

                                    +-----------+                                      
                                    | __start__ |                                      
                                    +-----------+                                      
                                          *                                            
                                          *                                            
                                          *                                            
                                   +------------+                                      
                                 **| supervisor |...                                   
                            *****  +------------+   .....                              
                     *******       ..          ..        .......                       
                *****            ..              ..             .....                  
            ****               .

In [3]:
PEDIDO = ("Hacé la due diligence de {nombre} para el comité de inversión: verificá las métricas del data room, "
          "calculá el costo real de los LLM y los unit economics, y marcá los riesgos que la startup no muestra.")


def ver_aportes(resultado):
    """Quién aportó qué, paso a paso (el campo `aportes` del estado)."""
    for a in resultado.estado["aportes"]:
        print(f"{a.paso:>2} | {a.agente:<12} | {a.accion:<16} | {a.detalle[:150]}")


def ver_uso(resultado):
    """Llamadas y tokens por agente (callback ContadorLLM)."""
    for agente, uso in resultado.uso_llm["por_agente"].items():
        print(f"{agente:<12} {uso['llamadas']:>3} llamadas · {uso['tokens_entrada']:>7} tokens de entrada · {uso['tokens_salida']:>5} de salida")
    total = resultado.uso_llm["total"]
    print(f"{'TOTAL':<12} {total['llamadas']:>3} llamadas · {total['tokens_entrada']:>7} tokens de entrada · {total['tokens_salida']:>5} de salida · {resultado.latencia_s} s")

## Caso 1 — VozActiva

El pitch dice *MRR de USD 120.000, margen bruto de 74 % y runway de 20 meses*. La letra chica del data room cuenta
otra historia. Abajo, el recorrido en vivo: cada línea `[paso N] supervisor →` es una decisión de delegación.

In [4]:
voz = await analizar(grafo, "vozactiva", PEDIDO.format(nombre="VozActiva"), config_analisis("demo-vozactiva", settings))
print("\nRecomendación preliminar:", voz.recomendacion)

[paso 1] supervisor → investigador: Busca en el data room de VozActiva y en fuentes de mercado las metricas principales de traccion, costos declarados y tamano de mercado, registrando cada dato con su cita literal.
           motivo: La investigacion esta vacia y la rubrica exige datos iniciales. Mandamos al investigador para registrar las metricas del data room.


    investigador: v1: 20 métricas ancladas, 4 hallazgos, 1 registros rechazados


[paso 2] supervisor → analista: Calcula el costo real de los LLM y los unit economics de VozActiva utilizando unicamente las metricas registradas por el investigador en el data room.
           motivo: La investigacion esta completa y aprobada, pero falta el analisis de unit economics y costos de LLM segun la rubrica.


    analista: v1: costo LLM USD 16.800/mes (18,7 % del MRR) · margen 41,9 % · LTV/CAC 0,93 · payback 16,6 m · runway 7,4 m · score 41,0 (no avanzar); 2 llamadas rechazadas por falta de respaldo


[paso 3] supervisor → critico: Audita el análisis v1 del analista, contrasta las métricas calculadas con los datos del investigador, busca letra chica y registra posibles red flags o correcciones.
           motivo: La rúbrica indica que la crítica no está vigente porque el crítico todavía no auditó el análisis v1. Debemos enviar al crítico para que audite los resultados.


    critico: v1 (auditó el análisis v1): 4 red flags (4 altas), 2 discrepancias declarado/calculado, 0 correcciones nuevas


[paso 4] supervisor → finalizar
           motivo: Todos los pasos obligatorios (investigación, análisis y crítica) se completaron con éxito y la rúbrica de suficiencia está totalmente satisfecha sin puntos bloqueantes [ ].
    validador: aprobada (intento 1, 0 advertencias)
    informe: NO AVANZAR (score 41,0/100 (banda: no avanzar); 4 red flags de severidad alta)

Recomendación preliminar: NO AVANZAR


### Quién aportó qué

El estado registra cada aporte (`aportes`, append-only) y cada especialista escribe solo su parcela.

In [5]:
ver_aportes(voz)

 1 | supervisor   | → investigador   | La investigacion esta vacia y la rubrica exige datos iniciales. Mandamos al investigador para registrar las metricas del data room.
 1 | investigador | investigación    | v1: 20 métricas ancladas, 4 hallazgos, 1 registros rechazados
 2 | supervisor   | → analista       | La investigacion esta completa y aprobada, pero falta el analisis de unit economics y costos de LLM segun la rubrica.
 2 | analista     | análisis         | v1: costo LLM USD 16.800/mes (18,7 % del MRR) · margen 41,9 % · LTV/CAC 0,93 · payback 16,6 m · runway 7,4 m · score 41,0 (no avanzar); 2 llamadas rec
 3 | supervisor   | → critico        | La rúbrica indica que la crítica no está vigente porque el crítico todavía no auditó el análisis v1. Debemos enviar al crítico para que audite los res
 3 | critico      | auditoría        | v1 (auditó el análisis v1): 4 red flags (4 altas), 2 discrepancias declarado/calculado, 0 correcciones nuevas
 4 | supervisor   | → finalizar      | Tod

### Proveniencia: cada número tiene fuente y cita

El investigador solo puede registrar un dato si la cita literal está en el fragmento y el valor aparece en la cita.
Las red flags de origen `contraste` las genera una regla determinística (declarado vs. calculado), sin LLM.

In [6]:
inv, ana, cri = voz.estado["investigacion"], voz.estado["analisis"], voz.estado["critica"]
print(f"Investigación v{inv.version}: {len(inv.metricas)} métricas, {len(inv.hallazgos)} hallazgos, {inv.rechazos} registros rechazados por el anclaje")
for nombre in ("mrr_usd", "margen_bruto_declarado_pct", "runway_declarado_meses", "otros_costos_directos_mensual_usd"):
    m = inv.metricas.get(nombre)
    if m:
        print(f"  {nombre} = {m.valor:g}  ←  \"{m.cita}\" [{m.fuente}]")
ue = ana.resultado("calcular_unit_economics")
print(f"\nAnálisis v{ana.version}: margen real {ue.get('margen_bruto_pct')} % · LTV/CAC {ue.get('ltv_cac')} · runway {ue.get('runway_meses')} meses · "
      f"{ana.rechazos} llamadas rechazadas por falta de respaldo")
print(f"\nCrítica v{cri.version} (auditó el análisis v{cri.version_analisis}):")
for r in cri.red_flags:
    print(f"  [{r.severidad}] {r.titulo} ({r.origen}) ← [{r.fuente}]")
for c in cri.correcciones:
    print(f"  corrección {c.id} → {c.agente_afectado} ({c.severidad}): {c.descripcion[:120]}")
print("\nValidación:")
for c in voz.estado["validacion"].checks:
    print(f"  {'⚠' if c.advertencia else ('✔' if c.ok else '✘')} {c.nombre}: {c.detalle}")

Investigación v1: 20 métricas, 4 hallazgos, 1 registros rechazados por el anclaje
  mrr_usd = 90000  ←  "MRR recurrente (contratos mensuales): USD 90.000" [vozactiva/metricas#1]
  margen_bruto_declarado_pct = 74  ←  "Margen bruto informado: 74 %" [vozactiva/finanzas#2]
  runway_declarado_meses = 20  ←  "Runway informado: 20 meses" [vozactiva/finanzas#4]
  otros_costos_directos_mensual_usd = 26000  ←  "Otros costos directos totales (soporte más voz y telefonía): USD 26.000 por mes" [vozactiva/finanzas#1]

Análisis v1: margen real 41.89 % · LTV/CAC 0.93 · runway 7.4 meses · 2 llamadas rechazadas por falta de respaldo

Crítica v1 (auditó el análisis v1):
  [alta] Runway artificial basado en financiamiento no firmado (critico) ← [vozactiva/finanzas#4]
  [alta] Alta concentración de clientes y riesgo de churn o descuento en cuenta principal (critico) ← [vozactiva/clientes-y-equipo#1]
  [alta] Margen bruto declarado inflado (contraste) ← [vozactiva/finanzas#2]
  [alta] Runway declarado mayor

In [7]:
ver_uso(voz)

analista       5 llamadas ·   16072 tokens de entrada ·   926 de salida
critico        5 llamadas ·   22489 tokens de entrada ·  1263 de salida
investigador   5 llamadas ·   31404 tokens de entrada ·  2049 de salida
supervisor     4 llamadas ·    3234 tokens de entrada ·   354 de salida
TOTAL         19 llamadas ·   73199 tokens de entrada ·  4592 de salida · 97.3 s


### El informe

Síntesis determinística (sin LLM) a partir de las parcelas validadas. También se guarda en `informes/vozactiva.md`.

In [8]:
voz.guardar(settings.informes_dir, settings.trazas_dir)
display(Markdown(voz.informe))

# Due diligence — VozActiva

Fecha: 2026-11-11 · Decisiones del supervisor: 4 · Validación: aprobada

## Recomendación preliminar: NO AVANZAR

Motivo: score 41,0/100 (banda: no avanzar); 4 red flags de severidad alta.

> Recomendación generada por el sistema a partir de datos anclados a sus fuentes. La decisión de invertir la toma el comité con aprobación humana explícita (la PE7 agrega ese paso al flujo).

## Métricas registradas (investigador)

| Métrica | Valor | Fuente | Cita |
|---|---|---|---|
| mrr_usd | 90.000 | vozactiva/metricas#1 | "MRR recurrente (contratos mensuales): USD 90.000" |
| clientes_activos | 20 | vozactiva/metricas#2 | "Clientes con contrato recurrente: 20" |
| crecimiento_mrr_mensual_pct | 4 | vozactiva/metricas#3 | "Crecimiento mensual del MRR recurrente (últimos 6 meses): 4 %" |
| churn_mensual_pct | 6,5 | vozactiva/metricas#4 | "Churn mensual de clientes recurrentes: 6,5 %" |
| gasto_comercial_mensual_usd | 78.000 | vozactiva/finanzas#3 | "Gasto mensual en ventas y marketing: USD 78.000" |
| clientes_nuevos_mes | 2,5 | vozactiva/metricas#4 | "Clientes recurrentes nuevos por mes (promedio de los últimos 6 meses): 2,5" |
| costo_infra_mensual_usd | 9.500 | vozactiva/finanzas#1 | "Infraestructura cloud (sin incluir el LLM): USD 9.500 por mes" |
| otros_costos_directos_mensual_usd | 26.000 | vozactiva/finanzas#1 | "Otros costos directos totales (soporte más voz y telefonía): USD 26.000 por mes" |
| margen_bruto_declarado_pct | 74 | vozactiva/finanzas#2 | "Margen bruto informado: 74 %" |
| burn_neto_mensual_usd | 95.000 | vozactiva/finanzas#4 | "Burn neto mensual: USD 95.000" |
| caja_usd | 700.000 | vozactiva/finanzas#4 | "Caja disponible al 31/10/2026: USD 700.000" |
| runway_declarado_meses | 20 | vozactiva/finanzas#4 | "Runway informado: 20 meses" |
| llamadas_llm_mes | 2.400.000 | vozactiva/tecnologia#2 | "Llamadas al LLM por mes: 2.400.000" |
| tokens_entrada_por_llamada | 1.800 | vozactiva/tecnologia#2 | "Tokens de entrada promedio por llamada: 1.800" |
| tokens_salida_por_llamada | 250 | vozactiva/tecnologia#2 | "Tokens de salida promedio por llamada: 250" |
| precio_entrada_usd_millon | 2,5 | vozactiva/tecnologia#3 | "Precio por millón de tokens de entrada: USD 2,50" |
| precio_salida_usd_millon | 10 | vozactiva/tecnologia#3 | "Precio por millón de tokens de salida: USD 10" |
| clientes_potenciales_tam | 9.800 | mercado/contact-centers-latam#1 | "Centros de contacto con 50 o más posiciones en la región: 9.800" |
| porcentaje_sam_pct | 57,1 | mercado/contact-centers-latam#1 | "México, Colombia y Argentina concentran 5.600 centros, el 57,1 % del total" |
| concentracion_top_cliente_pct | 46 | vozactiva/clientes-y-equipo#1 | "El cliente más grande, TelcoAndina, representa el 46 % del MRR recurrente" |

### Hallazgos cualitativos
- (clientes) El cliente más grande, TelcoAndina, representa el 46% del MRR recurrente y su contrato vence el 31/03/2027 pidiendo un 25% de descuento. [vozactiva/clientes-y-equipo#1]
- (equipo) La CTO Lucía Benítez tiene dedicación parcial del 50% y asesora a otra empresa. [vozactiva/clientes-y-equipo#2]
- (tecnologia) Dependencia de un proveedor único de LLM sin modelo alternativo probado en producción y con aumento del 40% en tokens de entrada desde 2027. [vozactiva/tecnologia#4]
- (riesgo) El runway informado de 20 meses incluye un préstamo puente de USD 1.200.000 no firmado. [vozactiva/finanzas#4]

## Análisis (analista, v1)

**Costo de LLM:** USD 16.800 por mes (18,7 % del MRR).

| ARPA | Margen bruto | CAC | LTV | LTV/CAC | Payback | Runway |
|---|---|---|---|---|---|---|
| USD 4.500 | 41,9 % | USD 31.200 | USD 29.000 | 0,93 | 16,6 meses | 7,4 meses |

**Scoring: 41,0/100 (no avanzar)**

| Criterio | Puntaje 0-5 | Peso | Aporte |
|---|---|---|---|
| unit_economics | 2,0 | 25 % | 10,0 |
| margen_bruto | 2,0 | 20 % | 8,0 |
| runway | 2,0 | 15 % | 6,0 |
| crecimiento | 2,0 | 15 % | 6,0 |
| equipo | 2,0 | 10 % | 4,0 |
| mercado | 3,0 | 10 % | 6,0 |
| concentracion | 1,0 | 5 % | 1,0 |

- Equipo: La CTO tiene dedicación parcial (50%), lo cual es un riesgo importante para una startup de IA.
- Mercado: El mercado de contact centers en LATAM es grande, pero la competencia y la alta concentración limitan el atractivo.

**Interpretación del analista:** Como analista financiero, he procesado las métricas registradas para **VozActiva** a través de las herramientas oficiales. 

### Resumen de Resultados Financieros y Unit Economics

- **Costo de LLM:** El costo mensual asciende a **$16,800 USD**, lo que representa un **18.67%** del MRR.
- **Margen Bruto Real:** Al incorporar los costos de infraestructura, otros costos directos y el consumo real de LLM, el margen bruto real se reduce al **41.89%** (muy por debajo del 74% declarado).
- **Eficiencia Comercial:** El CAC se ubica en **$31,200 USD**, con un LTV de **$29,000 USD**, arrojando un ratio **LTV/CAC de 0.93** y un **payback de 16.6 meses**.
- **Runway:** Con una caja de $700,000 USD y un burn neto mensual de $95,000 USD, el runway real es de **7.4 meses** (el valor declarado de 20 meses contemplaba un préstamo puente no firmado).
- **Scoring de Inversión:** La startup obtiene un puntaje de **41/100**, ubicándose en la banda de **"no avanzar"**. Esto refleja unit economics débiles (LTV/CAC menor a 1), un margen bruto erosionado por los costos de IA, alta concentración de clientes (46% en un solo cliente) y un runway crítico.

## Red flags (crítico)

- **[alta] Runway artificial basado en financiamiento no firmado**: El runway real es de 7.4 meses; contar con un préstamo no firmado distorsiona la salud financiera real de la empresa. Evidencia: "Nota 5: el runway informado incluye un préstamo puente de USD 1.200.000 que está en negociación y todavía no fue firmado" [vozactiva/finanzas#4]

- **[alta] Alta concentración de clientes y riesgo de churn o descuento en cuenta principal**: Una pérdida o reducción de este cliente impactaría drásticamente la supervivencia y los ingresos de la startup. Evidencia: "El cliente más grande, TelcoAndina, representa el 46 % del MRR recurrente.
- Los 3 clientes más grandes suman el 71 % del MRR recurrente.
- El contrato con TelcoAndina vence el 31/03/2027 y está en renegociación: el cliente pide un 25 % de descuento" [vozactiva/clientes-y-equipo#1]

- **[alta] Margen bruto declarado inflado** (regla de contraste): Margen bruto declarado 74,0 % vs. calculado con todos los costos directos 41,9 % (32,1 puntos de diferencia). Evidencia: "Margen bruto informado: 74 %" [vozactiva/finanzas#2]

- **[alta] Runway declarado mayor al real** (regla de contraste): Runway declarado 20 meses vs. 7,4 meses con la caja y el burn actuales. Evidencia: "Runway informado: 20 meses" [vozactiva/finanzas#4]

## Validación

- ✅ investigacion: 20 métricas ancladas a su fuente (mínimo 6) y 4 hallazgos con cita (mínimo 2).

- ✅ metricas_clave: Todas las métricas clave están registradas.

- ✅ correcciones_investigador: Sin correcciones altas pendientes.

- ✅ analisis_vigente: Análisis v1 completo y al día.

- ✅ correcciones_analista: Sin correcciones altas pendientes.

- ✅ critica_vigente: La crítica v1 auditó el análisis v1.

## Trazabilidad: quién aportó qué

| Paso | Agente | Acción | Detalle |
|---|---|---|---|
| 1 | supervisor | → investigador | La investigacion esta vacia y la rubrica exige datos iniciales. Mandamos al investigador para registrar las metricas del data room. |
| 1 | investigador | investigación | v1: 20 métricas ancladas, 4 hallazgos, 1 registros rechazados |
| 2 | supervisor | → analista | La investigacion esta completa y aprobada, pero falta el analisis de unit economics y costos de LLM segun la rubrica. |
| 2 | analista | análisis | v1: costo LLM USD 16.800/mes (18,7 % del MRR) · margen 41,9 % · LTV/CAC 0,93 · payback 16,6 m · runway 7,4 m · score 41,0 (no avanzar); 2 llamadas rechazadas por falta de respaldo |
| 3 | supervisor | → critico | La rúbrica indica que la crítica no está vigente porque el crítico todavía no auditó el análisis v1. Debemos enviar al crítico para que audite los resultados. |
| 3 | critico | auditoría | v1 (auditó el análisis v1): 4 red flags (4 altas), 2 discrepancias declarado/calculado, 0 correcciones nuevas |
| 4 | supervisor | → finalizar | Todos los pasos obligatorios (investigación, análisis y crítica) se completaron con éxito y la rúbrica de suficiencia está totalmente satisfecha sin puntos bloqueantes [ ]. |
| 4 | validador | validación | aprobada (intento 1, 0 advertencias) |

*Definiciones de métricas: 21 en el vocabulario de `state.py`.*


## Caso 2 — Contralia

Una startup con números sanos. Mismo grafo, mismo pedido.

In [9]:
contralia = await analizar(grafo, "contralia", PEDIDO.format(nombre="Contralia"), config_analisis("demo-contralia", settings))
contralia.guardar(settings.informes_dir, settings.trazas_dir)
sc = contralia.estado["analisis"].resultado("calcular_scoring")
ue = contralia.estado["analisis"].resultado("calcular_unit_economics")
print(f"\nRecomendación preliminar: {contralia.recomendacion} · score {sc.get('score')} ({sc.get('banda')})")
print(f"Margen {ue.get('margen_bruto_pct')} % · LTV/CAC {ue.get('ltv_cac')} · payback {ue.get('payback_meses')} meses · runway {ue.get('runway_meses')} meses")
for r in contralia.estado["critica"].red_flags:
    print(f"  [{r.severidad}] {r.titulo} ({r.origen})")
ver_uso(contralia)

[paso 1] supervisor → investigador: Revisa el data room y las fuentes de mercado de Contralia, registrando las métricas clave con sus citas literales y hallazgos cualitativos.
           motivo: El investigador aún no entregó datos, por lo que se debe iniciar el proceso con él según la rúbrica.


    investigador: v1: 21 métricas ancladas, 3 hallazgos, 0 registros rechazados


[paso 2] supervisor → analista: Calcula el costo real de los LLM, los unit economics y el tamaño de mercado utilizando exclusivamente las 21 métricas registradas por el investigador.
           motivo: La investigacion esta completa y sin correcciones pendientes, por lo que corresponde avanzar con el analista segun la rubrica para calcular unit economics y costos de LLM.


    analista: v1: costo LLM USD 6.240/mes (7,3 % del MRR) · margen 71,3 % · LTV/CAC 5,79 · payback 9,6 m · runway 24,0 m · score 92,0 (avanzar a comité); 0 llamadas rechazadas por falta de respaldo


17:58:11 | WARNING  | llm | Llamada al LLM fallida (GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}) -> se reintenta


[paso 3] supervisor → critico: Audita el análisis v1 realizado por el analista, contrasta lo declarado con lo calculado, busca la letra chica y registra posibles red flags o correcciones.
           motivo: El analista ya completó su trabajo y la rúbrica indica que la crítica no está vigente porque el crítico todavía no auditó el análisis [ ].


    critico: v1 (auditó el análisis v1): 0 red flags (0 altas), 0 discrepancias declarado/calculado, 0 correcciones nuevas


[paso 4] supervisor → finalizar
           motivo: Todos los pasos obligatorios (investigación, análisis y crítica) se completaron con éxito y la rúbrica de suficiencia está completamente satisfecha sin puntos bloqueantes [ ].
    validador: aprobada (intento 1, 0 advertencias)
    informe: AVANZAR A COMITÉ (score 92,0/100 (banda: avanzar a comité))

Recomendación preliminar: AVANZAR A COMITÉ · score 92.0 (avanzar a comité)
Margen 71.3 % · LTV/CAC 5.79 · payback 9.6 meses · runway 24.0 meses
analista       4 llamadas ·   12462 tokens de entrada ·   825 de salida
critico        6 llamadas ·   28218 tokens de entrada ·   633 de salida
investigador   5 llamadas ·   31630 tokens de entrada ·  1776 de salida
supervisor     5 llamadas ·    3232 tokens de entrada ·   346 de salida
TOTAL         20 llamadas ·   75542 tokens de entrada ·  3580 de salida · 103.1 s


## Caso 3 — Aula Viva: abstenerse también es una respuesta

El data room no tiene churn, CAC, burn ni costos de IA. La rúbrica pide buscar de nuevo; si después de dos búsquedas
el dato no está, queda como **no disponible** y el informe sale **SIN RECOMENDACIÓN** (se escala a un analista humano).

In [10]:
aula = await analizar(grafo, "aulaviva", PEDIDO.format(nombre="Aula Viva"), config_analisis("demo-aulaviva", settings))
aula.guardar(settings.informes_dir, settings.trazas_dir)
print("\nRecomendación preliminar:", aula.recomendacion)
print("Motivo:", next(a.detalle for a in aula.estado["aportes"] if a.agente == "informe"))
for c in aula.estado["validacion"].checks:
    print(f"  {'⚠' if c.advertencia else ('✔' if c.ok else '✘')} {c.nombre}: {c.detalle}")
ver_uso(aula)

[paso 1] supervisor → investigador: Busca en el data room y en fuentes de mercado las métricas clave de Aula Viva (usuarios, ingresos, costos de infraestructura y uso de LLM) registrando cada dato con su cita literal.
           motivo: El proceso recién empieza y la rúbrica muestra que el investigador todavía no entregó datos, por lo que es necesario asignarle la primera tarea.


    investigador: v1: 3 métricas ancladas, 3 hallazgos, 1 registros rechazados; faltan crecimiento_mrr_mensual_pct, churn_mensual_pct, concentracion_top_cliente_pct, gasto_comercial_mensual_usd, clientes_nuevos_mes, costo_infra_mensual_usd, otros_costos_directos_mensual_usd, burn_neto_mensual_usd, llamadas_llm_mes, tokens_entrada_por_llamada, tokens_salida_por_llamada, precio_entrada_usd_millon, precio_salida_usd_millon


[paso 2] supervisor → investigador: Buscá en el data room y completá las métricas faltantes: crecimiento_mrr_mensual_pct, churn_mensual_pct, concentracion_top_cliente_pct, gasto_comercial_mensual_usd, clientes_nuevos_mes, costo_infra_mensual_usd, otros_cos…
           motivo: Faltan métricas clave y llegar al mínimo de 6 requeridas por la rúbrica, por lo que debo enviar nuevamente al investigador a completarlas.


    investigador: v2: 5 métricas ancladas, 3 hallazgos, 1 registros rechazados; faltan crecimiento_mrr_mensual_pct, churn_mensual_pct, concentracion_top_cliente_pct, gasto_comercial_mensual_usd, clientes_nuevos_mes, costo_infra_mensual_usd, otros_costos_directos_mensual_usd, burn_neto_mensual_usd, llamadas_llm_mes, tokens_entrada_por_llamada, tokens_salida_por_llamada, precio_entrada_usd_millon, precio_salida_usd_millon


[paso 3] supervisor → analista: Calculá el costo de LLM, los unit economics y el tamaño de mercado utilizando exclusivamente las métricas registradas por el investigador.
           motivo: El investigador ya agotó o avanzó suficiente y la rúbrica indica que falta el análisis. Como el analista todavía no calculó (punto [x] bloqueante en analisis_vigente), corresponde enviarle la tarea.


    analista: v1: margen s/d · LTV/CAC s/d · payback s/d m · runway s/d m · score 12,0 (no avanzar); 1 llamadas rechazadas por falta de respaldo


[paso 4] supervisor → critico: Audita el análisis v1 del analista, contrasta lo declarado con lo calculado, busca letra chica y registra red flags y posibles errores.
           motivo: El analista ya procesó los datos, pero el crítico todavía no auditó el análisis (punto [ ] en la rúbrica). Debemos enviar al crítico.


    critico: v1 (auditó el análisis v1): 1 red flags (1 altas), 0 discrepancias declarado/calculado, 0 correcciones nuevas


18:00:26 | WARNING  | llm | Llamada al LLM fallida (GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}) -> se reintenta


[paso 5] supervisor → finalizar
           motivo: El investigador agotó sus intentos (2/3 pero con 2 búsquedas fallidas registradas en rúbrica) y la investigación tiene una salvedad [!] aceptable. El análisis y la crítica están vigentes y no hay correcciones pendientes, por lo que podemos finalizar el proceso.
    validador: aprobada (intento 1, 2 advertencias)
    informe: SIN RECOMENDACIÓN (información insuficiente: faltan 13 métricas clave (crecimiento_mrr_mensual_pct, churn_mensual_pct, concentracion_top_cliente_pct, gasto_comercial_mensual_usd, clientes_nuevos_mes, costo_infra_mensual_usd, otros_costos_directos_mensual_usd, burn_neto_mensual_usd, llamadas_llm_mes, tokens_entrada_por_llamada, tokens_salida_por_llamada, precio_entrada_usd_millon, precio_salida_usd_millon); pedirlas a la startup)

Recomendación preliminar: SIN RECOMENDACIÓN
Motivo: SIN RECOMENDACIÓN (información insuficiente: faltan 13 métricas clave (crecimiento_mrr_mensual_pct, churn_mensual_pct, concentracion_top

## Resumen de las tres corridas

In [11]:
print(f"{'Startup':<10} | {'Recomendación':<18} | {'Score':>5} | {'Pasos':>5} | {'Intentos (inv/ana/cri)':<22} | {'Llamadas':>8} | {'Tokens':>7} | Segundos")
for r in (voz, contralia, aula):
    sc = r.estado["analisis"].resultado("calcular_scoring") if r.estado["analisis"] else {}
    i = r.estado["intentos"]
    uso = r.uso_llm["total"]
    print(f"{r.startup:<10} | {r.recomendacion:<18} | {sc.get('score', '-'):>5} | {r.estado['pasos']:>5} | "
          f"{i['investigador']}/{i['analista']}/{i['critico']:<18} | {uso['llamadas']:>8} | {uso['tokens_entrada'] + uso['tokens_salida']:>7} | {r.latencia_s}")

Startup    | Recomendación      | Score | Pasos | Intentos (inv/ana/cri) | Llamadas |  Tokens | Segundos
vozactiva  | NO AVANZAR         |  41.0 |     4 | 1/1/1                  |       19 |   77791 | 97.3
contralia  | AVANZAR A COMITÉ   |  92.0 |     4 | 1/1/1                  |       20 |   79122 | 103.1
aulaviva   | SIN RECOMENDACIÓN  |  12.0 |     5 | 2/1/1                  |       20 |   53869 | 101.6


## Estado persistido en cada paso

El grafo compila con un checkpointer (`InMemorySaver`, con los contratos de `state.py` registrados en el serializador):
cada super-paso queda guardado y se puede inspeccionar. En la PE7 se reemplaza por Redis.

In [12]:
historial = [s async for s in grafo.aget_state_history(config_analisis("demo-vozactiva", settings))]
print(f"VozActiva: {len(historial)} checkpoints")
for s in reversed(historial[:-1]):
    if s.next:
        print(f"  después del paso {s.values.get('pasos')}: próximo nodo {s.next}")

VozActiva: 11 checkpoints
  después del paso 0: próximo nodo ('supervisor',)
  después del paso 1: próximo nodo ('investigador',)
  después del paso 1: próximo nodo ('supervisor',)
  después del paso 2: próximo nodo ('analista',)
  después del paso 2: próximo nodo ('supervisor',)
  después del paso 3: próximo nodo ('critico',)
  después del paso 3: próximo nodo ('supervisor',)
  después del paso 4: próximo nodo ('validador',)
  después del paso 4: próximo nodo ('informe',)
